# Qwen2.5-1.5B task-extraction SFT

Fine-tunes the model Xayra ships for on-device to-do extraction.

## Why

A 103-case evaluation harness put the current prompt-only baseline at **83.5%**, with two failures that three rounds of prompt work could not move:

| Failure | Score | Behaviour |
|---|---|---|
| Third-party attribution | **0/8 (0%)** | "The builder is coming Tuesday to look at the roof" becomes a task for the user |
| Zero-task refusal | **11/17 (65%)** | Observations and opinions produce invented tasks |

Few-shot examples fixed the specific cases in view and transferred to **none** of the unseen ones. That is a capacity limit, not a prompting problem.

## What this also buys

The production prompt is ~2,000 tokens, prefilled on every extraction. After SFT the behaviour lives in the weights, and the prompt drops to one sentence — which shrinks the KV-cache session file and the prefill cost with it.

## Runtime

Free Colab **T4**. Set `Runtime -> Change runtime type -> T4 GPU` before running. Roughly 15-25 minutes end to end for 450 samples over 3 epochs.

## 1. Dependencies

Unsloth pins compatible `trl`/`peft`/`xformers` builds itself; installing them loose alongside it is the usual cause of a broken Colab session.

In [ ]:
%%capture
import torch

major, _ = torch.cuda.get_device_capability()

!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

if major >= 8:
    # Ampere and newer (A100, L4): flash-attn and bf16 are available.
    !pip install -q --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    # T4 is Turing (sm75): no flash-attn, no bf16. Unsloth falls back to fp16.
    !pip install -q --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
import torch

print("GPU        :", torch.cuda.get_device_name(0))
print("Capability :", torch.cuda.get_device_capability())
print("VRAM (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("bf16       :", torch.cuda.is_bf16_supported())

## 2. Dataset

Upload `sft_qwen_task_extraction.jsonl`, produced by `scripts/dataset/generate_sft.py`.

Each row already carries a fully-formatted `text` field in Qwen2.5 ChatML, so no chat template is applied here. Re-templating an already-templated string is a common and silent corruption — it produces nested `<|im_start|>` markers and the model learns to emit them.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select sft_qwen_task_extraction.jsonl
DATASET_PATH = next(iter(uploaded))
print("Using:", DATASET_PATH)

In [ ]:
import json
from datasets import Dataset

rows = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8") if line.strip()]

empty = sum(1 for r in rows if not r["tasks"])
print(f"samples          : {len(rows)}")
print(f"empty-result     : {empty} ({empty / len(rows):.0%})")
print(f"distinct notes   : {len({r['note'] for r in rows})}")

# Guard against training on a corpus that collapsed into one class. Without
# positives the model learns to answer [] to everything, which scores well on
# the two failing categories and destroys the ones that already work.
assert 0.25 <= (1 - empty / len(rows)) <= 0.45, "positive/empty balance is off"

dataset = Dataset.from_list([{"text": r["text"]} for r in rows])
print()
print(dataset[0]["text"])

## 3. Model — QLoRA via Unsloth

`r=16` with `lora_alpha=16` (a 1:1 ratio) is deliberate for a corrective fine-tune: the goal is to suppress two specific behaviours, not to teach a new domain. A higher alpha relative to rank pushes harder on the base weights and risks damaging the extraction quality that is already at 100% on dates, STT and multi-task.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,      # None lets Unsloth pick fp16 on T4, bf16 on Ampere+
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,          # 0 is Unsloth's optimised path
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=20260921,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,}")

## 4. Train

3 epochs at `lr=2e-4`. With only ~450 short samples an epoch is fast, and three passes is enough to move behaviour without the model starting to recite training notes verbatim.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,   # samples are short and unrelated; packing would let
                         # one note's answer leak into the next note's context
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch size 8
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=20260921,
        output_dir="outputs",
        report_to="none",
    ),
)

stats = trainer.train()
print(stats)

## 5. Smoke test before export

Four prompts covering both target failures and both behaviours that must **not** regress. Worth thirty seconds here rather than discovering a collapsed model after a 1GB download.

Expected: `[]`, `[]`, a single task, two tasks.

In [ ]:
SYSTEM_PROMPT = (
    "You are an executive task extraction assistant. Extract actionable user "
    "tasks into the requested JSON schema. If no tasks exist for the user, "
    "return []."
)

PROBES = [
    ("The builder is coming Tuesday to look at the roof.", "[] — third party"),
    ("Feeling much better today than yesterday.", "[] — observation"),
    ("Remind me to call the dentist tomorrow.", "one task"),
    ("Book the flights and renew the travel insurance.", "two tasks"),
]

FastLanguageModel.for_inference(model)

for note, expectation in PROBES:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{note}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"{note}\n  expect: {expectation}\n  got   : {answer.strip()}\n")

## 6. Export to GGUF (q4_k_m)

Matches the quantization the app already ships, so on-device size and speed stay comparable. This step builds llama.cpp from source the first time and is the slowest cell in the notebook — 10-15 minutes is normal.

In [ ]:
model.save_pretrained_gguf(
    "qwen-task-extractor",
    tokenizer,
    quantization_method="q4_k_m",
)

!ls -lh qwen-task-extractor/*.gguf

In [ ]:
import glob
from google.colab import files

gguf = glob.glob("qwen-task-extractor/*.gguf")[0]
print("Downloading", gguf)
files.download(gguf)

## 7. Next steps, on your machine

```bash
# 1. Put the exported model where the harness looks
mv ~/Downloads/*.gguf models/qwen-task-extractor-q4_k_m.gguf

# 2. Score it against the frozen 103-case corpus, with the minimal prompt
XAYRA_EXTRACTION_PROMPT=minimal \
  npm run eval -- --model models/qwen-task-extractor-q4_k_m.gguf \
                  --corpus scripts/eval/corpus-full.jsonl
```

The bar to beat is **83.5% (86/103)**, and the two numbers that decide whether this worked are `third-party` (currently **0/8**) and `refusal` (currently **11/17**).

Watch the categories that already score 100% — dates, STT, multi-task, recurrence. A corrective fine-tune that fixes refusals by making the model answer `[]` more often will show up as regressions there, and that trade is not worth making.